# 02 : Génération des données opérationnelles (Maison Kurt)

Produit `data/operations.sqlite`, quatre tables normalisées : `magasins`,
`commandes`, `retours`, `remboursements`. Les règles de génération
reprennent celles écrites noir sur blanc dans les guides opérationnels
(`data/*.md`) : délai de retour de 30 jours pour un article standard, 14
jours pour un article soldé, remboursement sur carte bancaire plus lent
qu'un crédit sur carte cadeau, deux motifs de retour réservés aux
commandes en ligne.

Rien n'est mesuré sur de vraies commandes : les distributions choisies
ci-dessous ressemblent à des données d'exploitation plausibles, elles ne
reproduisent aucun chiffre réel de Maison Kurt, qui est une enseigne
fictive.

Le notebook `03_text_to_sql_operations.ipynb` réutilise la base produite
ici, donc il faut exécuter celui-ci en premier, ou après toute
modification des règles de génération.

## 1. Configuration

In [1]:
import sqlite3
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if (Path.cwd().parent / "data").exists() else Path.cwd()
DB_PATH = ROOT / "data" / "operations.sqlite"

SEED = 42
N_COMMANDES = 6000
DATE_DEBUT = date(2025, 1, 1)
DATE_SNAPSHOT = date(2025, 12, 15)   # "aujourd'hui" du jeu de données

# Valeurs manquantes non structurelles : un vrai système de caisse ou de
# gestion des retours perd occasionnellement un champ, indépendamment de la
# logique métier. Deux causes plausibles sont simulées, à des taux faibles
# pour ne pas dominer les analyses qui suivent :
P_MANQUANT_MOYEN_PAIEMENT = 0.01   # panne/synchronisation du terminal de caisse
P_MANQUANT_MOTIF_RETOUR = 0.02     # conseiller n'ayant pas renseigné le motif
P_DOUBLON_COMMANDE = 0.003         # échec de confirmation suivi d'une relance

MAGASINS = [
    (1, "Maison Kurt Paris Haussmann", "Paris"),
    (2, "Maison Kurt Lyon Part-Dieu", "Lyon"),
    (3, "Maison Kurt Marseille Prado", "Marseille"),
    (4, "Maison Kurt Bordeaux Chartrons", "Bordeaux"),
    (5, "Maison Kurt Lille Centre", "Lille"),
    (6, "Maison Kurt Toulouse Capitole", "Toulouse"),
    (7, "Maison Kurt Nantes Centre", "Nantes"),
    (8, "Maison Kurt Strasbourg Centre", "Strasbourg"),
]

CANAUX = ["magasin", "en ligne - livraison", "en ligne - click&collect"]
P_CANAL = [0.45, 0.40, 0.15]
SLA_JOURS = {"magasin": 0, "en ligne - click&collect": 2, "en ligne - livraison": 5}

MOTIFS_COMMUNS = ["ne convient pas", "taille inadaptée", "article défectueux", "changement d'avis"]
MOTIFS_EN_LIGNE = ["colis endommagé à réception", "article incorrect reçu"]


def _bornes(x: np.ndarray, lo: float, hi: float) -> np.ndarray:
    return np.clip(x, lo, hi)

## 2. Commandes

Un magasin, un canal (magasin / livraison à domicile / Click&Collect), un
type d'article (standard / soldé), un moyen de paiement, un montant. Le
délai réel de livraison tourne autour du SLA affiché par canal.

In [2]:
def generer_commandes(rng: np.random.Generator) -> pd.DataFrame:
    magasin_id = rng.integers(1, len(MAGASINS) + 1, N_COMMANDES)
    canal = rng.choice(CANAUX, N_COMMANDES, p=P_CANAL)
    type_article = rng.choice(["standard", "soldé"], N_COMMANDES, p=[0.85, 0.15])
    moyen_paiement = rng.choice(["carte bancaire", "carte cadeau"], N_COMMANDES, p=[0.75, 0.25]).astype(object)
    moyen_paiement[rng.random(N_COMMANDES) < P_MANQUANT_MOYEN_PAIEMENT] = None

    jours_ecoules = rng.integers(0, (DATE_SNAPSHOT - DATE_DEBUT).days + 1, N_COMMANDES)
    date_commande = np.array([DATE_DEBUT + timedelta(days=int(j)) for j in jours_ecoules])

    montant = np.where(
        type_article == "standard",
        _bornes(rng.normal(65, 25, N_COMMANDES), 10, 250),
        _bornes(rng.normal(35, 15, N_COMMANDES), 8, 150),
    ).round(2)

    delai_sla = np.array([SLA_JOURS[c] for c in canal])

    # Délai réel de livraison, autour du SLA affiché par canal.
    delai_reel = np.empty(N_COMMANDES)
    for i, c in enumerate(canal):
        if c == "magasin":
            delai_reel[i] = 0
        elif c == "en ligne - click&collect":
            delai_reel[i] = _bornes(rng.normal(1.8, 1.0), 0, 8)
        else:  # en ligne - livraison
            delai_reel[i] = _bornes(rng.normal(4.3, 1.8), 1, 16)
    delai_reel = delai_reel.round().astype(int)

    date_livraison_prevue = np.array(
        [dc + timedelta(days=int(d)) for dc, d in zip(date_commande, delai_reel)]
    )
    livree = date_livraison_prevue <= DATE_SNAPSHOT

    statut = np.where(canal == "magasin", "livrée", np.where(livree, "livrée", "en cours"))
    delai_traitement = np.where((canal == "magasin") | livree, delai_reel, np.nan)
    en_retard = np.where(
        (canal != "magasin") & livree,
        delai_traitement > delai_sla,
        np.nan,
    )

    return pd.DataFrame({
        "commande_id": np.arange(1, N_COMMANDES + 1),
        "magasin_id": magasin_id,
        "canal": canal,
        "date_commande": [d.isoformat() for d in date_commande],
        "type_article": type_article,
        "moyen_paiement": moyen_paiement,
        "montant": montant,
        "statut": statut,
        "delai_traitement_jours": delai_traitement,
        "delai_sla_jours": delai_sla,
        "en_retard": en_retard,
    })

## 3. Doublons (échec de confirmation, relance)

La caisse ou la plateforme ne renvoie pas l'accusé de réception à temps,
l'opération est relancée, et les deux tentatives aboutissent quand même :
deux commandes identiques en tout point (même magasin, même date, même
montant...) sont créées, sauf l'identifiant technique, qui, lui, ne peut
pas être partagé.

In [3]:
def injecter_doublons(rng: np.random.Generator, commandes: pd.DataFrame) -> pd.DataFrame:
    a_dupliquer = commandes[rng.random(len(commandes)) < P_DOUBLON_COMMANDE].copy()
    if a_dupliquer.empty:
        return commandes
    a_dupliquer["commande_id"] = np.arange(
        commandes["commande_id"].max() + 1, commandes["commande_id"].max() + 1 + len(a_dupliquer)
    )
    return pd.concat([commandes, a_dupliquer], ignore_index=True)

## 4. Retours

Seules les commandes déjà `livrée` sont éligibles. Le taux de retour
dépend du canal, et la combinaison "soldé + livraison à domicile" (achat
impulsif, sans essayage) a un taux nettement plus élevé que les autres
combinaisons canal/type. Le motif n'est pas toujours renseigné (incident
de saisie), et environ 15% des retours tombent hors délai contractuel :
le cas "dépassement du délai" documenté dans le guide retours/échanges,
section 6.

In [4]:
def generer_retours(rng: np.random.Generator, commandes: pd.DataFrame) -> pd.DataFrame:
    eligibles = commandes[commandes["statut"] == "livrée"].copy()

    p_retour = pd.Series(0.10, index=eligibles.index)
    p_retour += np.where(eligibles["canal"] == "en ligne - click&collect", 0.05, 0.0)
    p_retour += np.where(eligibles["canal"] == "en ligne - livraison", 0.15, 0.0)
    p_retour += np.where(
        (eligibles["type_article"] == "soldé") & (eligibles["canal"] == "en ligne - livraison"),
        0.20,
        np.where(eligibles["type_article"] == "soldé", 0.03, 0.0),
    )
    tire = rng.random(len(eligibles))
    retournees = eligibles[tire < p_retour.values].copy()

    n = len(retournees)
    en_ligne_mask = retournees["canal"] != "magasin"

    motifs = []
    for is_en_ligne in en_ligne_mask:
        pool = MOTIFS_COMMUNS + (MOTIFS_EN_LIGNE if is_en_ligne else [])
        poids = [0.32, 0.28, 0.22, 0.18] if not is_en_ligne else [0.24, 0.21, 0.17, 0.13, 0.14, 0.11]
        motifs.append(rng.choice(pool, p=poids))
    motifs = np.array(motifs, dtype=object)
    motifs[rng.random(n) < P_MANQUANT_MOTIF_RETOUR] = None

    fenetre = np.where(retournees["type_article"].values == "standard", 30, 14)
    delta_jours = rng.integers(1, (fenetre + 10) + 1)
    dates_commande = pd.to_datetime(retournees["date_commande"]).values
    date_retour = dates_commande + delta_jours.astype("timedelta64[D]")
    dans_delai = delta_jours <= fenetre

    delai_traitement = _bornes(rng.normal(2.0, 1.5, n), 0, 10).round().astype(int)

    return pd.DataFrame({
        "retour_id": np.arange(1, n + 1),
        "commande_id": retournees["commande_id"].values,
        "date_retour": pd.to_datetime(date_retour).strftime("%Y-%m-%d"),
        "motif": motifs,
        "delai_traitement_jours": delai_traitement,
        "dans_delai": dans_delai.astype(int),
    })

## 5. Remboursements

Un remboursement suit un retour accepté. Le taux de conformité dépend du
délai et du motif (un article défectueux reste remboursable même hors
délai, conformément au guide). Le délai de remboursement dépend du moyen
de paiement d'origine, quand il est connu. S'il ne l'est pas (données
manquantes sur la commande), il est tiré dans le mélange global plutôt
que supposé.

In [5]:
def generer_remboursements(rng: np.random.Generator, commandes: pd.DataFrame, retours: pd.DataFrame) -> pd.DataFrame:
    fusion = retours.merge(
        commandes[["commande_id", "moyen_paiement", "montant"]], on="commande_id", how="left"
    )

    p_conforme = np.where(
        fusion["motif"] == "article défectueux",
        0.97,
        np.where(fusion["dans_delai"] == 1, 0.95, 0.20),
    )
    tire = rng.random(len(fusion))
    conformes = fusion[tire < p_conforme].copy()

    n = len(conformes)
    moyen = conformes["moyen_paiement"].values
    est_cb = moyen == "carte bancaire"
    est_cc = moyen == "carte cadeau"
    inconnu = ~(est_cb | est_cc)

    delai = np.empty(n)
    delai[est_cb] = _bornes(rng.normal(4.0, 1.5, est_cb.sum()), 1, 12)
    delai[est_cc] = _bornes(rng.normal(1.0, 0.5, est_cc.sum()), 0, 4)
    if inconnu.any():
        tire_cb_inconnu = rng.random(inconnu.sum()) < 0.75
        delai[inconnu] = np.where(
            tire_cb_inconnu,
            _bornes(rng.normal(4.0, 1.5, inconnu.sum()), 1, 12),
            _bornes(rng.normal(1.0, 0.5, inconnu.sum()), 0, 4),
        )
    delai = delai.round().astype(int)

    date_retour = pd.to_datetime(conformes["date_retour"]).values
    date_remboursement = date_retour + delai.astype("timedelta64[D]")

    return pd.DataFrame({
        "remboursement_id": np.arange(1, n + 1),
        "retour_id": conformes["retour_id"].values,
        "moyen_paiement": conformes["moyen_paiement"].values,
        "montant": conformes["montant"].values,
        "date_remboursement": pd.to_datetime(date_remboursement).strftime("%Y-%m-%d"),
        "delai_jours": delai,
    })

## 6. Exécution et écriture

Génère les quatre tables, dans l'ordre des dépendances (`commandes` avant
`retours`, `retours` avant `remboursements`), puis écrit
`data/operations.sqlite`.

In [6]:
rng = np.random.default_rng(SEED)

magasins = pd.DataFrame(MAGASINS, columns=["magasin_id", "nom", "ville"])
commandes = generer_commandes(rng)
commandes = injecter_doublons(rng, commandes)
retours = generer_retours(rng, commandes)
commandes.loc[commandes["commande_id"].isin(retours["commande_id"]), "statut"] = "retournée"
remboursements = generer_remboursements(rng, commandes, retours)

DB_PATH.parent.mkdir(parents=True, exist_ok=True)
with sqlite3.connect(DB_PATH) as con:
    magasins.to_sql("magasins", con, index=False, if_exists="replace")
    commandes.to_sql("commandes", con, index=False, if_exists="replace")
    retours.to_sql("retours", con, index=False, if_exists="replace")
    remboursements.to_sql("remboursements", con, index=False, if_exists="replace")

print(f"Base écrite : {DB_PATH}")
print(f"  magasins        : {len(magasins)}")
print(f"  commandes       : {len(commandes)}")
print(f"  retours         : {len(retours)}")
print(f"  remboursements  : {len(remboursements)}")

Base écrite : C:\Users\jboul\Desktop\Dev\Linkdin\demo\retail-operations-rag\data\operations.sqlite
  magasins        : 8
  commandes       : 6024
  retours         : 1080
  remboursements  : 830
